# Credit Limit Analysis (Starter)

This notebook performs a short exploratory analysis and ranks features by impact on `credit_limit` using absolute linear-model coefficients.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv('../data/credit_data.csv')
df.head()

In [ ]:
# Basic cleaning
df = df.drop_duplicates().copy()
numeric_cols = df.select_dtypes(include='number').columns.tolist()
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
df.isna().sum().sort_values(ascending=False).head()

In [ ]:
# Summary patterns
df.describe(include='all').T

In [ ]:
# Visualizations
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['credit_limit'], kde=True, ax=axes[0])
axes[0].set_title('Distribution of Credit Limit')
sns.scatterplot(data=df, x='income', y='credit_limit', hue='region', ax=axes[1])
axes[1].set_title('Income vs Credit Limit')
plt.tight_layout()

In [ ]:
# Feature impact ranking using absolute coefficient values
target_col = 'credit_limit'
feature_cols = [c for c in numeric_cols if c not in [target_col, 'customer_id']]
X = df[feature_cols]
y = df[target_col]

model = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])
model.fit(X, y)

coefs = model.named_steps['regressor'].coef_
feature_impact_ranking = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': coefs
})
feature_impact_ranking['abs_coefficient'] = feature_impact_ranking['coefficient'].abs()
feature_impact_ranking = feature_impact_ranking.sort_values('abs_coefficient', ascending=False).reset_index(drop=True)
feature_impact_ranking

In [ ]:
feature_impact_ranking.to_csv('../data/feature_impact_ranking.csv', index=False)
print('Saved: ../data/feature_impact_ranking.csv')